# Vivado API

This notebook walks through the complete Vivado API workflow using a small but realistic multi-file VHDL design: a 4-bit synchronous counter driving four LEDs on a Basys3 board. The design is intentionally minimal so the focus stays on the API mechanics rather than the HDL.  The steps are:

1. Inspect the VHDL source files
2. Upload all files to the server
3. Create a synthesis job and wait for it to complete
4. Retrieve synthesis logs, utilization/timing/power reports
5. Run a behavioral simulation and retrieve its log

## Configuration

Set the server address and the bearer token that protects all API endpoints. Two header dictionaries are prepared: one for requests that carry a JSON body (`json_headers`) and one for everything else (`headers`).

In [1]:
import json
import time
from pathlib import Path

import requests

BASE_URL = "http://ricpc:8000"
API_KEY  = "43567543243653856"

# Plain auth header — used for GET requests and POST requests without a body
headers = {"Authorization": f"Bearer {API_KEY}"}

# Auth + content-type — used for POST requests that send a JSON payload
json_headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

### Health check

`GET /health` is the only unauthenticated endpoint.  It confirms the server is running, reports the path to the Vivado binary, and tells us whether authentication is enabled.  Always call this first to make sure the server is reachable and that Vivado is installed where the server expects it.

In [2]:
r = requests.get(f"{BASE_URL}/health")
print("Status:", r.status_code)
print(json.dumps(r.json(), indent=2))

Status: 200
{
  "status": "ok",
  "vivado_path": "C:/Xilinx/2025.1/Vivado/bin/vivado.bat",
  "vivado_available": true,
  "workspace_dir": "C:/vivado_api",
  "auth_enabled": false
}


## VHDL design

The example is split across three source files and one testbench:

- **`src/counter.vhd`** — a 4-bit synchronous up-counter with active-high reset. This is the only stateful element in the design; it holds the current count in a registered signal and increments it on every rising clock edge.

- **`src/led_driver.vhd`** — a purely combinational block that maps the 4-bit count bus to the four LED outputs. On the Basys3 the LEDs are active-high, so the signal is passed through without inversion.

- **`src/top.vhd`** — the structural top level.  It instantiates both components and connects them through an internal signal called `count`.

- **`tb/tb_top.vhd`** — the self-checking testbench. It generates a 100 MHz clock, holds reset for four cycles, then runs the counter for 20 cycles and asserts the exact LED value expected on each cycle.

The cells below read each file from disk and display it.

In [3]:
# Helper: print a VHDL file with a separator so it is easy to read in the notebook output
def show_vhdl(path: str) -> None:
    print(f"{'─' * 60}")
    print(f"  {path}")
    print(f"{'─' * 60}")
    print(Path(path).read_text())

### The counter component

This file implements a 4-bit synchronous up-counter with an active-high reset using sequential logic and unsigned arithmetic from numeric_std.

In [4]:
show_vhdl("src/counter.vhd")

────────────────────────────────────────────────────────────
  src/counter.vhd
────────────────────────────────────────────────────────────
-- counter.vhd
-- 4-bit synchronous up-counter with active-high reset.

library ieee;
use ieee.std_logic_1164.all;
use ieee.numeric_std.all;

entity counter is
    port (
        clk   : in  std_logic;
        rst   : in  std_logic;
        count : out std_logic_vector(3 downto 0)
    );
end entity counter;

architecture rtl of counter is
    signal cnt : unsigned(3 downto 0) := (others => '0');
begin
    process(clk)
    begin
        if rising_edge(clk) then
            if rst = '1' then
                cnt <= (others => '0');
            else
                cnt <= cnt + 1;
            end if;
        end if;
    end process;

    count <= std_logic_vector(cnt);
end architecture rtl;



### The led driver component

This module maps a 4-bit input data bus directly to four active-high LED outputs.

In [5]:
show_vhdl("src/led_driver.vhd")

────────────────────────────────────────────────────────────
  src/led_driver.vhd
────────────────────────────────────────────────────────────
-- led_driver.vhd
-- Drives four active-high LEDs from a 4-bit data bus.

library ieee;
use ieee.std_logic_1164.all;

entity led_driver is
    port (
        data : in  std_logic_vector(3 downto 0);
        leds : out std_logic_vector(3 downto 0)
    );
end entity led_driver;

architecture rtl of led_driver is
begin
    leds <= data;   -- Basys3 LEDs are active-high.
end architecture rtl;



### The Top level component

This top-level module connects the 4-bit counter output to the LED driver, exposing clock, reset, and LED signals at the FPGA boundary.

In [6]:
show_vhdl("src/top.vhd")

────────────────────────────────────────────────────────────
  src/top.vhd
────────────────────────────────────────────────────────────
-- top.vhd
-- Top-level entity: wires a 4-bit counter to the LED driver.

library ieee;
use ieee.std_logic_1164.all;

entity top is
    port (
        clk  : in  std_logic;
        rst  : in  std_logic;
        leds : out std_logic_vector(3 downto 0)
    );
end entity top;

architecture rtl of top is

    component counter is
        port (
            clk   : in  std_logic;
            rst   : in  std_logic;
            count : out std_logic_vector(3 downto 0)
        );
    end component counter;

    component led_driver is
        port (
            data : in  std_logic_vector(3 downto 0);
            leds : out std_logic_vector(3 downto 0)
        );
    end component led_driver;

    signal count : std_logic_vector(3 downto 0);

begin

    cnt_inst : counter
        port map (
            clk   => clk,
            rst   => rst,
            count 

### The testbench

This self-checking testbench verifies that the top-level design correctly resets and that the LED outputs follow the expected 4-bit counter value.

In [7]:
show_vhdl("tb/tb_top.vhd")

────────────────────────────────────────────────────────────
  tb/tb_top.vhd
────────────────────────────────────────────────────────────
-- tb_top.vhd
-- Self-checking testbench for the top-level counter + LED driver design.
-- Verifies that after reset the LEDs track the counter value.

library ieee;
use ieee.std_logic_1164.all;
use ieee.numeric_std.all;

entity tb_top is
end entity tb_top;

architecture sim of tb_top is

    signal clk  : std_logic := '0';
    signal rst  : std_logic := '1';
    signal leds : std_logic_vector(3 downto 0);

    component top is
        port (
            clk  : in  std_logic;
            rst  : in  std_logic;
            leds : out std_logic_vector(3 downto 0)
        );
    end component top;

begin

    -- 100 MHz clock
    clk <= not clk after 5 ns;

    dut : top
        port map (
            clk  => clk,
            rst  => rst,
            leds => leds
        );

    stim : process
        variable expected : std_logic_vector(3 downto 0);
   

### Upload sources

`POST /upload` accepts one or more files in a multipart form and returns a `file_id` for each one.  These IDs are stable references that the server stores permanently, they are used in subsequent calls to identify which
files belong to a job. All five files are uploaded in a single request: the three design sources, the testbench, and the XDC constraints.  File handles are closed in a `finally` block to avoid resource leaks even if the request fails.

In [8]:
src_files = sorted(Path("src").glob("*.vhd"))
tb_files  = sorted(Path("tb").glob("*.vhd"))
xdc_files = sorted(Path("constraints").glob("*.xdc"))

all_files = list(src_files) + list(tb_files) + list(xdc_files)

print("Files to upload:")
for f in all_files:
    print(" ", f)

multipart = []
handles   = []

try:
    for f in all_files:
        fh = open(f, "rb")
        handles.append(fh)
        multipart.append(("files", (f.name, fh, "text/plain")))

    r = requests.post(
        f"{BASE_URL}/upload",
        headers=headers,
        files=multipart,
    )
finally:
    for fh in handles:
        fh.close()

upload_result = r.json()
print("\nStatus:", r.status_code)
print(json.dumps(upload_result, indent=2))

Files to upload:
  src/counter.vhd
  src/led_driver.vhd
  src/top.vhd
  tb/tb_top.vhd
  constraints/top.xdc

Status: 200
{
  "files": [
    {
      "file_id": "bbdd7899-2b08-4bf7-ad55-b3737a47632c",
      "filename": "counter.vhd",
      "size_bytes": 696
    },
    {
      "file_id": "25e07bbf-fe05-4647-b916-92d875386703",
      "filename": "led_driver.vhd",
      "size_bytes": 497
    },
    {
      "file_id": "9abb319b-a1d7-4885-86dd-ded031361987",
      "filename": "top.vhd",
      "size_bytes": 1016
    },
    {
      "file_id": "057d4ba7-f743-4f1c-ba9e-d9757633cbc5",
      "filename": "tb_top.vhd",
      "size_bytes": 1553
    },
    {
      "file_id": "6ffda5a6-381b-45a0-af13-7ee442ae33ee",
      "filename": "top.xdc",
      "size_bytes": 676
    }
  ]
}


The upload response lists every file with its assigned `file_id`.  We build a dictionary so later cells can look up IDs by filename rather than by index, which makes the intent clearer when constructing job payloads.

In [9]:
file_map = {item["filename"]: item["file_id"] for item in upload_result["files"]}
file_map

{'counter.vhd': 'bbdd7899-2b08-4bf7-ad55-b3737a47632c',
 'led_driver.vhd': '25e07bbf-fe05-4647-b916-92d875386703',
 'top.vhd': '9abb319b-a1d7-4885-86dd-ded031361987',
 'tb_top.vhd': '057d4ba7-f743-4f1c-ba9e-d9757633cbc5',
 'top.xdc': '6ffda5a6-381b-45a0-af13-7ee442ae33ee'}

## Synthesis

`POST /jobs` registers a new job on the server.  The payload specifies:

- `board_id` — selects the target FPGA part from the server's board catalog (`GET /boards` lists all available options).
- `top_entity` — the name of the VHDL entity that Vivado should treat as the design root during synthesis.
- `file_ids` — the design source files.  **Only VHDL files go here.**  The XDC constraints and the testbench are intentionally excluded: Vivado's `read_vhdl` command rejects non-VHDL files, and the testbench is only needed for simulation. The job starts in `pending` state.  Nothing runs yet.

In [10]:
payload = {
    "board_id":   "basys3",
    "top_entity": "top",
    "file_ids": [
        file_map["counter.vhd"],
        file_map["led_driver.vhd"],
        file_map["top.vhd"],
    ],
}

r = requests.post(f"{BASE_URL}/jobs", headers=json_headers, json=payload)

job_info = r.json()
print("Status:", r.status_code)
print(json.dumps(job_info, indent=2))

# Save the job ID for all subsequent calls
JOB_ID = job_info["job_id"]
print("\nJOB_ID:", JOB_ID)

Status: 201
{
  "job_id": "0b64e966-45a9-42c2-bee5-49e74b58a409",
  "status": "pending",
  "board_id": "basys3",
  "top_entity": "top",
  "created_at": "2026-06-22T14:43:57.632321Z",
  "updated_at": "2026-06-22T14:43:57.632321Z",
  "error": null
}

JOB_ID: 0b64e966-45a9-42c2-bee5-49e74b58a409


### Launch synthesis

`POST /jobs/{job_id}/synthesize` kicks off Vivado synthesis in the background. The server immediately returns with the job in `synthesizing` state; the actual work runs asynchronously so this call returns quickly. Vivado is invoked in batch mode with a generated TCL script that reads the uploaded VHDL sources, runs `synth_design`, writes utilization / timing / power reports, and saves a checkpoint (`.dcp`) for later inspection.

In [11]:
r = requests.post(
    f"{BASE_URL}/jobs/{JOB_ID}/synthesize",
    headers=headers,
)

print("Status:", r.status_code)
print(json.dumps(r.json(), indent=2))

Status: 200
{
  "job_id": "0b64e966-45a9-42c2-bee5-49e74b58a409",
  "status": "synthesizing",
  "board_id": "basys3",
  "top_entity": "top",
  "created_at": "2026-06-22T14:43:57.632321Z",
  "updated_at": "2026-06-22T14:44:02.603259Z",
  "error": null
}


`GET /jobs/{job_id}` returns the current job status.  We poll every five seconds until the status reaches either `synthesized` (success) or `failed` (error). The server updates the job status in real time as Vivado runs, so this polling loop is the simplest way to track progress without needing a more complex event system.

In [12]:
while True:
    r      = requests.get(f"{BASE_URL}/jobs/{JOB_ID}", headers=headers)
    status = r.json()["status"]
    print(status)
    if status in ("synthesized", "failed"):
        break
    time.sleep(5)

synthesizing
synthesizing
synthesizing
synthesizing
synthesizing
synthesizing
synthesizing
synthesized


### Log

`GET /jobs/{job_id}/log?type=synthesis` returns the full Vivado log as plain text. This is the same file Vivado would write to disk in a normal project flow. It is the primary diagnostic tool when synthesis fails: look for lines starting with `ERROR:` or `CRITICAL WARNING:` to understand what went wrong. We print only the last 80 lines here to keep the output manageable.

In [13]:
r = requests.get(
    f"{BASE_URL}/jobs/{JOB_ID}/log",
    params={"type": "synthesis"},
    headers=headers,
)

print("Status:", r.status_code)
if r.ok:
    lines = r.text.splitlines()
    print(f"[... {max(0, len(lines) - 80)} lines omitted ...]")
    print("\n".join(lines[-80:]))
else:
    print(r.text)

Status: 200
[... 149 lines omitted ...]
---------------------------------------------------------------------------------
---------------------------------------------------------------------------------
Finished Renaming Generated Ports : Time (s): cpu = 00:00:09 ; elapsed = 00:00:17 . Memory (MB): peak = 1508.387 ; gain = 1010.609
---------------------------------------------------------------------------------
---------------------------------------------------------------------------------
Start Handling Custom Attributes
---------------------------------------------------------------------------------
---------------------------------------------------------------------------------
Finished Handling Custom Attributes : Time (s): cpu = 00:00:09 ; elapsed = 00:00:17 . Memory (MB): peak = 1508.387 ; gain = 1010.609
---------------------------------------------------------------------------------
---------------------------------------------------------------------------------
Start R

### Results

`GET /jobs/{job_id}/synthesis` returns the structured synthesis outcome: elapsed time, parsed utilization / timing / power numbers, the last few lines of the log, and any error message.  Unlike the raw log endpoint, this is easy to process programmatically, useful for automated regression or CI pipelines.

In [14]:
r = requests.get(f"{BASE_URL}/jobs/{JOB_ID}/synthesis", headers=headers)

print("Status:", r.status_code)
print(json.dumps(r.json(), indent=2))

Status: 200
{
  "status": "synthesized",
  "elapsed_sec": 35.78996680001728,
  "utilization": {
    "luts": 5,
    "lut_ram": 0,
    "flip_flops": 4,
    "dsps": 0,
    "brams": 0,
    "ios": 6,
    "lut_pct": 0.02,
    "ff_pct": null,
    "dsp_pct": 0.0,
    "bram_pct": 0.0
  },
  "timing": {
    "wns_ns": null,
    "tns_ns": null,
    "whs_ns": null,
    "ths_ns": null,
    "timing_met": null
  },
  "power": {
    "total_power_w": 3.926,
    "dynamic_power_w": 3.843,
    "static_power_w": 0.083,
    "junction_temp_c": 44.6,
    "confidence": "Low"
  },
  "log_tail": "-------------------------------\nStart Handling Custom Attributes\n---------------------------------------------------------------------------------\n---------------------------------------------------------------------------------\nFinished Handling Custom Attributes : Time (s): cpu = 00:00:09 ; elapsed = 00:00:17 . Memory (MB): peak = 1508.387 ; gain = 1010.609\n---------------------------------------------------------

### Report

`GET /jobs/{job_id}/reports` bundles all three Vivado report files in a single response: structured fields for quick comparison and the raw report text for detailed inspection. A few notes on what to expect from a synthesis-only run:

- **LUT count** is marked with an asterisk in the report (`Slice LUTs*`) because the number is an estimate before place-and-route.  The API strips the asterisk and reports the estimate as-is.
- **Timing** (WNS / TNS) will be `null` unless an XDC file with a clock constraint was included in the synthesis `file_ids`. Meaningful timing closure numbers only come out of the implementation (place-and-route) stage.
- **Power** is a rough estimate at synthesis stage; it becomes more accurate after implementation and activity annotation.

In [15]:
r = requests.get(f"{BASE_URL}/jobs/{JOB_ID}/reports", headers=headers)

print("Status:", r.status_code)

if r.ok:
    reports = r.json()

    util   = reports.get("utilization") or {}
    timing = reports.get("timing") or {}
    power  = reports.get("power") or {}

    print("\n-- Utilization --")
    print(f"  LUTs        : {util.get('luts')} ({util.get('lut_pct')} %)")
    print(f"  Flip-flops  : {util.get('flip_flops')} ({util.get('ff_pct')} %)")
    print(f"  DSPs        : {util.get('dsps')} ({util.get('dsp_pct')} %)")
    print(f"  BRAMs       : {util.get('brams')} ({util.get('bram_pct')} %)")

    print("\n-- Timing --")
    print(f"  WNS         : {timing.get('wns_ns')} ns")
    print(f"  TNS         : {timing.get('tns_ns')} ns")
    print(f"  Timing met  : {timing.get('timing_met')}")

    print("\n-- Power --")
    print(f"  Total       : {power.get('total_power_w')} W")
    print(f"  Dynamic     : {power.get('dynamic_power_w')} W")
    print(f"  Static      : {power.get('static_power_w')} W")
    print(f"  Junction T  : {power.get('junction_temp_c')} °C")
else:
    print(r.json())

Status: 200

-- Utilization --
  LUTs        : 5 (0.02 %)
  Flip-flops  : 4 (None %)
  DSPs        : 0 (0.0 %)
  BRAMs       : 0 (0.0 %)

-- Timing --
  WNS         : None ns
  TNS         : None ns
  Timing met  : None

-- Power --
  Total       : 3.926 W
  Dynamic     : 3.843 W
  Static      : 0.083 W
  Junction T  : 44.6 °C


## Simulation

`POST /jobs/{job_id}/simulate` runs a behavioral (pre-implementation) simulation using Vivado's XSim simulator.  The job must already be in `synthesized` state. The payload identifies the testbench:

- `testbench_file_id` — the file ID of the testbench VHDL file.
- `testbench_top` — the name of the testbench entity that XSim should elaborate.
- `runtime` — how long to run the simulation (e.g. `300ns`, `1us`, `10ms`).

Internally the server creates a temporary on-disk Vivado project (required by `launch_simulation`), compiles all design sources plus the testbench, and runs XSim for the requested duration. Like synthesis, this runs in the background.

In [16]:
sim_payload = {
    "testbench_file_id": file_map["tb_top.vhd"],
    "testbench_top":     "tb_top",
    # 300 ns covers 4 reset cycles + 20 counter cycles at 100 MHz (10 ns period)
    "runtime":           "300ns",
}

r = requests.post(
    f"{BASE_URL}/jobs/{JOB_ID}/simulate",
    headers=json_headers,
    json=sim_payload,
)

print("Status:", r.status_code)
print(json.dumps(r.json(), indent=2))

Status: 200
{
  "job_id": "0b64e966-45a9-42c2-bee5-49e74b58a409",
  "status": "simulating",
  "board_id": "basys3",
  "top_entity": "top",
  "created_at": "2026-06-22T14:43:57.632321Z",
  "updated_at": "2026-06-22T14:48:22.264162Z",
  "error": null
}


Same polling pattern as synthesis. Terminal states are `done` (success) and `failed` (error).

In [17]:
while True:
    r      = requests.get(f"{BASE_URL}/jobs/{JOB_ID}", headers=headers)
    status = r.json()["status"]
    print(status)
    if status in ("done", "failed"):
        break
    time.sleep(5)

simulating
simulating
done


### Result

`GET /jobs/{job_id}/simulation` returns elapsed time and any error message. A successful run will show `"status": "done"` and `"error": null`.  If the testbench raised a VHDL `severity failure` assertion, XSim exits with a non-zero code and the error appears here.

In [18]:
r = requests.get(f"{BASE_URL}/jobs/{JOB_ID}/simulation", headers=headers)

print("Status:", r.status_code)
print(json.dumps(r.json(), indent=2))

Status: 200
{
  "status": "done",
  "elapsed_sec": 14.163034499855712,
  "log_tail": ": [USF-XSim-61] Executing 'COMPILE and ANALYZE' step in 'C:/vivado_api/jobs/0b64e966-45a9-42c2-bee5-49e74b58a409/sim_project/sim_project.sim/sim_1/behav/xsim'\n\"xvhdl --incr --relax -prj tb_top_vhdl.prj\"\nINFO: [VRFC 10-163] Analyzing VHDL file \"C:/vivado_api/uploads/bbdd7899-2b08-4bf7-ad55-b3737a47632c/counter.vhd\" into library xil_defaultlib\nINFO: [VRFC 10-3107] analyzing entity 'counter'\nINFO: [VRFC 10-163] Analyzing VHDL file \"C:/vivado_api/uploads/25e07bbf-fe05-4647-b916-92d875386703/led_driver.vhd\" into library xil_defaultlib\nINFO: [VRFC 10-3107] analyzing entity 'led_driver'\nINFO: [VRFC 10-163] Analyzing VHDL file \"C:/vivado_api/uploads/9abb319b-a1d7-4885-86dd-ded031361987/top.vhd\" into library xil_defaultlib\nINFO: [VRFC 10-3107] analyzing entity 'top'\nINFO: [VRFC 10-163] Analyzing VHDL file \"C:/vivado_api/uploads/057d4ba7-f743-4f1c-ba9e-d9757633cbc5/tb_top.vhd\" into library xil

### Log

`GET /jobs/{job_id}/log?type=simulation` returns the XSim log.  This includes the compile step (VHDL analysis), elaboration, and the simulation run itself. Any `report` or `assert` statements in the testbench appear here, making it the right place to confirm that the design behaved as expected.

In [19]:
r = requests.get(
    f"{BASE_URL}/jobs/{JOB_ID}/log",
    params={"type": "simulation"},
    headers=headers,
)

print("Status:", r.status_code)
if r.ok:
    lines = r.text.splitlines()
    print(f"[... {max(0, len(lines) - 80)} lines omitted ...]")
    print("\n".join(lines[-80:]))
else:
    print(r.text)

Status: 200
[... 33 lines omitted ...]
# read_vhdl -vhdl2008 "C:/vivado_api/uploads/057d4ba7-f743-4f1c-ba9e-d9757633cbc5/tb_top.vhd"
# set_property top      $tb_top      [get_filesets sim_1]
# set_property top_lib  xil_defaultlib [get_filesets sim_1]
# update_compile_order  -fileset sim_1
# puts "INFO: Launching behavioral simulation ..."
INFO: Launching behavioral simulation ...
# launch_simulation -simset sim_1 -mode behavioral
Command: launch_simulation  -simset sim_1 -mode behavioral
INFO: [Vivado 12-12493] Simulation top is 'tb_top'
INFO: [Vivado 12-5682] Launching behavioral simulation in 'C:/vivado_api/jobs/0b64e966-45a9-42c2-bee5-49e74b58a409/sim_project/sim_project.sim/sim_1/behav/xsim'
INFO: [SIM-utils-51] Simulation object is 'sim_1'
INFO: [SIM-utils-72] Using boost library from 'C:/Xilinx/2025.1/Vivado/tps/boost_1_72_0'
INFO: [SIM-utils-54] Inspecting design source files for 'tb_top' in fileset 'sim_1'...
INFO: [USF-XSim-97] Finding global include files...
INFO: [USF-XSim-9